# Week 3 — Baseline Forecast Evaluation

**Goal:** Measure how good (or bad) the moving-average baseline is using two
standard forecasting error metrics — **MAPE** and **RMSE** — both overall and
per product category. These numbers are the bar that ARIMA must beat next.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

DAILY_PATH = Path('..') / 'data' / 'processed' / 'supply_chain_daily.csv'
TARGET = 'units_sold'
HORIZON = 30   # days held out for testing
WINDOW = 7     # moving-average window

daily = pd.read_csv(DAILY_PATH, parse_dates=['date'])
daily = daily.sort_values(['product_category', 'date']).reset_index(drop=True)
print('Loaded shape:', daily.shape)
print('Date range:', daily['date'].min().date(), '->', daily['date'].max().date())

## 1. Rebuild the baseline forecast

Same train/test split (last 30 days held out per category) and the same
flat moving-average baseline used in `forecasting.ipynb`, so evaluation is
self-contained.

In [ ]:
def train_test_split(frame, horizon=HORIZON):
    train_parts, test_parts = [], []
    for cat, sub in frame.groupby('product_category'):
        sub = sub.sort_values('date')
        train_parts.append(sub.iloc[:-horizon])
        test_parts.append(sub.iloc[-horizon:])
    return pd.concat(train_parts, ignore_index=True), pd.concat(test_parts, ignore_index=True)

def moving_average_forecast(train_frame, test_frame, window=WINDOW):
    parts = []
    for cat, sub in test_frame.groupby('product_category'):
        sub = sub.sort_values('date').copy()
        last_window = (train_frame[train_frame['product_category'] == cat]
                       .sort_values('date')[TARGET].tail(window))
        sub['forecast_ma'] = last_window.mean()
        parts.append(sub[['date', 'product_category', TARGET, 'forecast_ma']])
    return pd.concat(parts, ignore_index=True)

train, test = train_test_split(daily)
baseline = moving_average_forecast(train, test)
baseline['forecast_ma'] = baseline['forecast_ma'].round(2)
print('Baseline forecast rows:', len(baseline))
baseline.head()

## 2. Error metrics

- **MAPE** (Mean Absolute Percentage Error) — average error as a % of the actual
  value; easy to read but blows up when actuals are near zero, so we mask zeros.
- **RMSE** (Root Mean Squared Error) — error in the same units as demand; punishes
  big misses more than small ones.

In [ ]:
def mape(actual, predicted):
    """Mean Absolute Percentage Error (%), ignoring rows where actual == 0."""
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    mask = actual != 0
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100)

def rmse(actual, predicted):
    """Root Mean Squared Error, in the same units as the target."""
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return float(np.sqrt(np.mean((actual - predicted) ** 2)))

print('Overall baseline MAPE %:', round(mape(baseline[TARGET], baseline['forecast_ma']), 2))
print('Overall baseline RMSE  :', round(rmse(baseline[TARGET], baseline['forecast_ma']), 2))

## 3. Per-category error table

Break the error down by product so we can see which categories the baseline
handles well and which ones it struggles with.

In [ ]:
rows = []
for cat, sub in baseline.groupby('product_category'):
    rows.append({
        'product_category': cat,
        'MAPE_%': round(mape(sub[TARGET], sub['forecast_ma']), 2),
        'RMSE': round(rmse(sub[TARGET], sub['forecast_ma']), 2),
    })

baseline_scores = pd.DataFrame(rows).sort_values('MAPE_%').reset_index(drop=True)
baseline_scores

## 4. Visualize the error by category

A quick bar chart of MAPE per category makes the weak spots obvious at a glance.

In [ ]:
order = baseline_scores.sort_values('MAPE_%')
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(order['product_category'], order['MAPE_%'], color='steelblue')
ax.set_title('Baseline moving-average error (MAPE %) by category')
ax.set_ylabel('MAPE %')
ax.set_xlabel('product category')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()